# Penguin Species Prediction with scikit-learn

This notebook predicts a penguin's species from two measurements, `bill_length_mm` and `flipper_length_mm`. It follows the complete small-model workflow: load and clean a CSV, encode species names, split the data, build a scaling/KNN pipeline, inspect decision boundaries, evaluate held-out predictions, and decode a new prediction.

## Environment and execution

From `90_resources/courses/fastapi/ml_fastapi`, install the notebook environment:

```sh
uv sync --locked --group notebook
```

Open this notebook in VS Code, select this `ml_fastapi` project's `.venv/bin/python` as the kernel, and run the cells from top to bottom. `pandas`, `numpy`, `matplotlib`, and `scikit-learn` provide the data/model/plotting tools; `ipykernel` runs the notebook and `nbclient` supports automated execution. The notebook uses only scikit-learn; this project also includes TensorFlow for the workshop's MNIST inference examples.

The API links below use the **scikit-learn 1.5.2** documentation to match the version pinned in this project. The CSV remains in `90_resources/courses/fastapi/ml_fastapi/data/`; the loading cell finds it when the kernel starts in the repository root, `ml_fastapi` root, or `example` directory.

## Origin

This notebook belongs to the FastAPI ML inference workflow and stays in `ml_fastapi/example/`, alongside the workshop's API examples. The [workshop](../README.md#exercise-2-prediction-on-demand) references [Cheukting/FastAPI-ml-demo](https://github.com/Cheukting/FastAPI-ml-demo/blob/main/penguins_predict.ipynb). This local version adds explanations and adapts the CSV lookup; the original modeling cells and sample inputs are retained. The workshop's [MIT license](../LICENSE) remains with the source materials.

## 1. Load the data and remove missing rows

`pd.read_csv` reads the penguin table into a DataFrame. The `pathlib` lookup searches the current working directory and its parents for the existing CSV, so moving the notebook does not require a second copy of the data.

`data.dropna()` removes every row with a missing value in **any column**, including columns not used by this model. This preserves the original example's cleaning rule; limiting missing-value removal to the two features and target would produce a different training dataset. The final `data` expression displays the cleaned table.

Supporting APIs: [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [DataFrame.dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html), and [pathlib.Path](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path).

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

for repo_root in (Path.cwd(), *Path.cwd().parents):
    data_path = repo_root / "90_resources/courses/fastapi/ml_fastapi/data/penguins.csv"
    if data_path.is_file():
        break
else:
    raise FileNotFoundError("Start the notebook kernel inside the FromPython2Agent repository.")

data = pd.read_csv(data_path)
data = data.dropna()
data

## 2. Define features, encode species, and split the samples

`X` contains two columns in a fixed order: bill length first, flipper length second. Its shape is `(n_samples, 2)`. `y` is a one-dimensional array of integer class IDs with shape `(n_samples,)`.

[`LabelEncoder`](https://scikit-learn.org/1.5/modules/generated/sklearn.preprocessing.LabelEncoder.html) converts the **target** species names to IDs. [`fit`](https://scikit-learn.org/1.5/modules/generated/sklearn.preprocessing.LabelEncoder.html#sklearn.preprocessing.LabelEncoder.fit) records the class names in `le.classes_`, and [`transform`](https://scikit-learn.org/1.5/modules/generated/sklearn.preprocessing.LabelEncoder.html#sklearn.preprocessing.LabelEncoder.transform) applies that mapping. These IDs identify categories; they do not imply that one species is numerically greater than another. Here the encoder records the complete label vocabulary before splitting; it does not estimate feature statistics.

[`train_test_split`](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html) returns training and test features and targets. With neither size specified, the default test fraction is 25%. `stratify=y` approximately preserves each species' proportion, and `random_state=0` makes the split reproducible for the same data and environment. Feature scaling is fitted later, using only the training set.

In [ ]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

X = data[["bill_length_mm", "flipper_length_mm"]]
le.fit(data["species"])
y = le.transform(data["species"])
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

## 3. Construct the scaling and classification pipeline

[`Pipeline`](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html) chains two named steps:

| Step | What it does | Why it matters here |
| --- | --- | --- |
| `scaler`: [`StandardScaler`](https://scikit-learn.org/1.5/modules/generated/sklearn.preprocessing.StandardScaler.html) | Transforms each feature using its training mean and standard deviation, `z = (x - mean) / std`. | KNN uses distances, so features with different spreads should be placed on comparable scales. |
| `knn`: [`KNeighborsClassifier`](https://scikit-learn.org/1.5/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) | Predicts from the labels of the 11 nearest training samples. | `n_neighbors=11` controls the size of the voting neighborhood. |

This cell only creates `clf`; fitting happens in the next cell. [`Pipeline.fit`](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline.fit) fits the scaler on `X_train`, transforms those samples, and fits KNN. Later predictions reuse those same scaling statistics, keeping test data out of the scaler's fit.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

clf = Pipeline(
    steps=[("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=11))]
)

## 4. Fit two weighting schemes and visualize their decisions

The loop fits the same pipeline twice. `weights="uniform"` gives each of the 11 neighbors an equal vote; `weights="distance"` gives closer neighbors more influence. [`Pipeline.set_params`](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline.set_params) uses `knn__weights` to change the `weights` parameter inside the named `knn` step. `clf[-1]` accesses that final estimator.

[`DecisionBoundaryDisplay.from_estimator`](https://scikit-learn.org/1.5/modules/generated/sklearn.inspection.DecisionBoundaryDisplay.html#sklearn.inspection.DecisionBoundaryDisplay.from_estimator) evaluates the fitted pipeline on a two-dimensional grid. `X_test` defines the displayed grid range; `response_method="predict"` colors it with predicted class IDs, and `plot_method="pcolormesh"` draws the regions. The black-edged scatter points use **all** cleaned samples in `X`, including training and test samples, so this plot is a visual explanation rather than a held-out performance metric.

**State after the loop:** `clf` holds the final fit with `weights="distance"`. The confusion matrices and new predictions below all use that fitted configuration. This loop illustrates the two settings; it does not select a winner through cross-validation.

Supporting plotting API: [matplotlib.pyplot.subplots](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html).

In [ ]:
import matplotlib.pyplot as plt

from sklearn.inspection import DecisionBoundaryDisplay

_, axs = plt.subplots(ncols=2, figsize=(12, 5))

for ax, weights in zip(axs, ("uniform", "distance")):
    clf.set_params(knn__weights=weights).fit(X_train, y_train)
    disp = DecisionBoundaryDisplay.from_estimator(
        clf,
        X_test,
        response_method="predict",
        plot_method="pcolormesh",
        xlabel="bill_length_mm",
        ylabel="flipper_length_mm",
        shading="auto",
        alpha=0.5,
        ax=ax,
    )
    scatter = disp.ax_.scatter(X.iloc[:, 0], X.iloc[:, 1], c=y, edgecolors="k")
    disp.ax_.legend(
        scatter.legend_elements()[0],
        le.classes_,
        loc="lower left",
        title="Classes",
    )
    _ = disp.ax_.set_title(
        f"3-Class classification\n(k={clf[-1].n_neighbors}, weights={weights!r})"
    )

plt.show()

## 5. Evaluate on the held-out test set

[`ConfusionMatrixDisplay.from_estimator`](https://scikit-learn.org/1.5/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_estimator) calls the fitted pipeline's prediction method on `X_test` and compares the results with `y_test`. Rows represent true species; columns represent predicted species. `display_labels=le.classes_` supplies readable names in encoder order.

- With `normalize=None`, each entry is a sample count. Diagonal entries are correct predictions; off-diagonal entries show which species are confused.
- With `normalize="true"`, each row is divided by that true class's total. Each populated row sums to 1, and its diagonal value is that class's recall.

Both figures describe the same distance-weighted model and test split. `np.set_printoptions(precision=2)` changes how NumPy prints numbers; it does not change model calculations or the confusion matrix values.

Supporting API: [numpy.set_printoptions](https://numpy.org/doc/stable/reference/generated/numpy.set_printoptions.html).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import ConfusionMatrixDisplay

np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
titles_options = [
    ("Confusion matrix, without normalization", None),
    ("Normalized confusion matrix", "true"),
]
for title, normalize in titles_options:
    disp = ConfusionMatrixDisplay.from_estimator(
        clf,
        X_test,
        y_test,
        display_labels=le.classes_,
        cmap=plt.cm.Blues,
        normalize=normalize,
    )
    disp.ax_.set_title(title)

    print(title)
    print(disp.confusion_matrix)

plt.show()

## 6. Predict from a named input row

The dictionary becomes a one-row DataFrame with the same two feature names and order as `X`. [`Pipeline.predict`](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline.predict) scales that row and passes it to KNN, returning an array of encoded class IDs, not species names or probabilities.

The original example uses `bill_length_mm=195` and `flipper_length_mm=195`. The bill length lies outside this dataset's observed range. KNN still returns a nearest-neighbor label, but that output does not establish a reliable prediction for an unusual input. For an ordinary-input experiment, you can later try a bill length such as 45 and a flipper length such as 200; the original values are retained here for comparison.

In [ ]:
param = {
                "bill_length_mm": [195],
                "flipper_length_mm": [195]
            }
clf.predict(pd.DataFrame(param))

## 7. Inspect the test input type

`type(X_test)` confirms that splitting a pandas DataFrame preserves the DataFrame container. It retains feature names as well as values, which helps keep training and inference inputs consistent. This inspection does not train or modify the model.

Related API: [`train_test_split` return values](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html#sklearn.model_selection.train_test_split).

In [ ]:
type(X_test)

## 8. Decode a prediction into a species name

`[[195, 195]]` is a two-dimensional input with shape `(1, 2)`: one penguin and two features. A flat list would not express that sample/feature structure. The positions must match the training order, bill length followed by flipper length.

[`LabelEncoder.inverse_transform`](https://scikit-learn.org/1.5/modules/generated/sklearn.preprocessing.LabelEncoder.html#sklearn.preprocessing.LabelEncoder.inverse_transform) converts the predicted IDs back into species names; `[0]` selects the first sample's name.

This original list-based example can emit a feature-name warning because `StandardScaler` was fitted using a DataFrame. The named DataFrame from Section 6 preserves the feature schema and is the clearer input format for an API. The out-of-range bill-length caveat still applies.

In [ ]:
result = clf.predict([[195,195]])
le.inverse_transform(result)[0]

## Connecting this notebook to FastAPI

The notebook produces a fitted `clf` and a matching label encoder `le`. An API can reuse these objects for inference, build a named one-row DataFrame from validated measurements, and return the decoded species string. Training belongs in an explicit initialization/training stage rather than every prediction request.

Continue with [Part 1, Exercise 2: prediction on demand](../README.md#exercise-2-prediction-on-demand). The blank cell below is available for your own experiments; it does not contain an API implementation.